# 1. Import Required Libraries
Import pandas, numpy, os, and scikit-learn preprocessing modules for data manipulation and feature engineering.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# 2. Load the Merged Data
Read the merged CSV data file using pandas for further processing.

In [2]:
input_path = os.path.join("..", "integration", "merged_data.csv")
df = pd.read_csv(input_path)
df.head()

C:\Users\drket\AppData\Local\Temp\ipykernel_24356\1406275813.py:2: DtypeWarning: Columns (1,2,4,10,16,17,18,19,20,21,22,39,53,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


,ID,State,City,location,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,"Madhurangan Apartment ,Ambegaon, Pune",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,"Manganahalli Sriram Layout ,Ullal Uppana...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,"sona Building,Bhayandar West, Mumbai",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,"Sec 2 Pooja apartment Bhosari ,Indrayani Nag...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,"A N SWAGATH,Gubbalala, Subramanyapura,Bangalore",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Drop Irrelevant Columns
Remove columns that are not useful for modeling, such as IDs, timestamps, and location coordinates.

In [3]:
irrelevant_cols = [
    "ID", "timestamp", "date", "name", "address", "location", "Unnamed: 0", "serial_no", "lat", "lon", "latitude", "longitude"
]
irrelevant_cols = [col for col in irrelevant_cols if col in df.columns]
df = df.drop(columns=irrelevant_cols)
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 4. Add New Features
Create new features such as price per square foot, bedrooms per square foot, polynomial features, total rooms, property age, binned property age, and inflation-adjusted price.

In [4]:
# Price per sqft, bedrooms per sqft, polynomial features
if "area" in df.columns and "bedrooms" in df.columns:
    df["price_per_sqft"] = df["price"] / df["area"]
    df["bedrooms_per_sqft"] = df["bedrooms"] / df["area"]
    df["area_sq"] = df["area"] ** 2
    df["bedrooms_sq"] = df["bedrooms"] ** 2

# Total rooms
if "bedrooms" in df.columns and "bathrooms" in df.columns and "total_rooms" not in df.columns:
    df["total_rooms"] = df["bedrooms"] + df["bathrooms"]

# Property age, binned age, inflation-adjusted price
if "year_built" in df.columns:
    current_year = pd.Timestamp.now().year
    df["property_age"] = current_year - df["year_built"]
    df["property_age_bin"] = pd.cut(df["property_age"], bins=[0, 5, 15, 30, 100], labels=["new", "mid", "old", "very_old"])
    inflation_index = {year: 1 + 0.05 * (current_year - year) for year in df["year_built"].unique()}
    df["inflation_adjusted_price"] = df.apply(lambda x: x["price"] * inflation_index.get(x["year_built"], 1), axis=1)
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 5. Environmental Features
Add or impute environmental features like air quality, noise level, crime rate, and water quality.

In [5]:
for col in ["air_quality", "noise_level", "crime_rate", "water_quality"]:
    if col not in df.columns:
        df[col] = np.nan
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 6. Interaction Features
Create interaction features between area and air quality, and price per square foot and crime rate.

In [6]:
if "area" in df.columns and "air_quality" in df.columns:
    df["area_x_air_quality"] = df["area"] * df["air_quality"]
if "price_per_sqft" in df.columns and "crime_rate" in df.columns:
    df["price_per_sqft_x_crime_rate"] = df["price_per_sqft"] * df["crime_rate"]
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 7. Log Transforms for Skewed Features
Apply log transformations to skewed features such as area, price, and price per square foot.

In [7]:
for col in ["area", "price", "price_per_sqft"]:
    if col in df.columns:
        df[f"log_{col}"] = np.log1p(df[col])
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 8. Boolean Flags
Create a boolean flag for luxury properties based on price per square foot quantile.

In [8]:
if "price_per_sqft" in df.columns:
    df["is_luxury"] = (df["price_per_sqft"] > df["price_per_sqft"].quantile(0.9)).astype(int)
df.head()

,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 9. Categorical Encoding
Encode categorical features using one-hot encoding for low cardinality and label encoding for high cardinality.

In [9]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "property_age_bin"]
for col in categorical_cols:
    if df[col].nunique() < 10:
        dummies = pd.get_dummies(df[col], prefix=col)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(columns=[col])
    else:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
df.head()

,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False


# 10. Statistical Aggregations
Compute mean and standard deviation of price per location if location data is available.

In [10]:
if "location" in df.columns and "price" in df.columns:
    df["mean_price_location"] = df.groupby("location")["price"].transform("mean")
    df["std_price_location"] = df.groupby("location")["price"].transform("std")
df.head()

,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False


# 11. Fill Missing Values
Fill missing values in the dataset using the median strategy for numerical columns.

In [11]:
df = df.fillna(df.median(numeric_only=True))
df.head()

,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,6.0,...,False,False,False,False,True,False,False,False,False,False
1,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,6.0,...,False,False,False,False,True,False,False,False,False,False
2,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,6.0,...,False,False,False,True,False,False,False,False,False,False
3,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,6.0,...,False,False,False,False,True,False,False,False,False,False
4,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,6.0,...,False,False,False,True,False,False,False,False,False,False


# 12. Save the Feature Engineered Data
Save the processed DataFrame to a CSV file for further modeling and analysis.

In [12]:
output_path = os.path.join("feature_engineered.csv")
df.to_csv(output_path, index=False)
print(f"Feature engineered data saved to {output_path}")

Feature engineered data saved to feature_engineered.csv


# Expert Feature Engineering for Price Prediction
As a feature engineering expert, I will:
1. Consolidate all price-related columns into a single target variable
2. Remove irrelevant columns that don't contribute to price prediction
3. Create meaningful features from the existing data
4. Filter for Bangalore properties only for consistency
5. Handle categorical encoding properly based on data distribution

In [13]:
# Expert Feature Engineering: Data Analysis and Cleaning
# Load the already processed data
input_path = os.path.join("..", "integration", "merged_data.csv")
df = pd.read_csv(input_path)

print("Original dataset shape:", df.shape)
print("\nColumns in dataset:")
print(df.columns.tolist())

# First, let's examine price-related columns
price_columns = [col for col in df.columns if 'price' in col.lower()]
print(f"\nPrice-related columns: {price_columns}")

# Check for missing values
print(f"\nMissing values per column:")
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

# Data is already filtered for Bangalore, so no need to filter again
print(f"\nTotal rows (already filtered for Bangalore): {len(df)}")

print("\nFirst few rows:")
df.head()

Original dataset shape: (21386, 67)

Columns in dataset:
['ID', 'State', 'City', 'location', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Furnished_Status_encoded', 'Public_Transport_Accessibility_encoded', 'Parking_Space_encoded', 'Security_encoded', 'Availability_Status_encoded', 'Name', 'Property Title', 'Price', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony', 'balcony_encoded', 'Price_numeric', 'feature_importance', 'Date_x', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'Month_x', 'Year_x', 'AQI_Bucket_encoded', 'Year_y', 'Month_y', 'Day', 'Night', 'DayLimit', 'NightLimit', 'Date_y', 'StationEncoded', 'DayExcess', 'NightExce

C:\Users\drket\AppData\Local\Temp\ipykernel_24356\37246463.py:4: DtypeWarning: Columns (1,2,4,10,16,17,18,19,20,21,22,39,53,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


,ID,State,City,location,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,"Madhurangan Apartment ,Ambegaon, Pune",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,"Manganahalli Sriram Layout ,Ullal Uppana...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,"sona Building,Bhayandar West, Mumbai",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,"Sec 2 Pooja apartment Bhosari ,Indrayani Nag...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,"A N SWAGATH,Gubbalala, Subramanyapura,Bangalore",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Consolidate Price Columns into Single Target Variable
# Work with existing processed data

def extract_price_from_text(price_text):
    """Extract numeric price from text format like '₹23.0 L' or '₹1.5 Cr'"""
    if pd.isna(price_text):
        return np.nan
    
    price_str = str(price_text).replace('₹', '').replace(',', '').strip()
    
    if 'L' in price_str:
        # Lakhs format
        numeric_part = float(price_str.replace('L', '').strip())
        return numeric_part * 100000  # Convert lakhs to rupees
    elif 'Cr' in price_str:
        # Crores format
        numeric_part = float(price_str.replace('Cr', '').strip())
        return numeric_part * 10000000  # Convert crores to rupees
    else:
        try:
            return float(price_str)
        except:
            return np.nan

# Create consolidated price column based on available data
if 'Price_numeric' in df.columns and df['Price_numeric'].notna().any():
    df['target_price'] = df['Price_numeric']
    print("Using Price_numeric as target_price")
elif 'Price_in_Lakhs' in df.columns and df['Price_in_Lakhs'].notna().any():
    df['target_price'] = df['Price_in_Lakhs'] * 100000
    print("Using Price_in_Lakhs as target_price")
elif 'Price' in df.columns:
    df['target_price'] = df['Price'].apply(extract_price_from_text)
    print("Using Price text column as target_price")
else:
    print("No suitable price column found!")

# Show target price statistics
if 'target_price' in df.columns:
    print(f"\nTarget price statistics:")
    print(df['target_price'].describe())
    print(f"Non-null values: {df['target_price'].notna().sum()}")
else:
    print("Warning: Could not create target_price column!")

Using Price_numeric as target_price

Target price statistics:
count    1.451800e+04
mean     1.067542e+07
std      1.867913e+07
min      1.000000e+00
25%      3.700000e+06
50%      6.500000e+06
75%      1.140000e+07
max      8.400000e+08
Name: target_price, dtype: float64
Non-null values: 14518


In [15]:
# Remove Irrelevant Columns for Price Prediction
# Check current columns first
print("Current columns in dataset:")
print(df.columns.tolist())
print(f"\nTotal columns: {len(df.columns)}")

# Define columns to remove (only if they exist)
columns_to_remove = [
    'Unnamed: 0', 'index', 'ID', 'id', 'Id',  # Index columns
    'Property_Link', 'Property_URL', 'url',   # URLs
    'Property_Image', 'images',               # Images
    'Description', 'description', 'desc',    # Text descriptions
    'Posted_Date', 'date_posted',            # Posting dates
    'Contact_Info', 'contact', 'phone',      # Contact info
    'Agent_Name', 'agent', 'seller'          # Agent details
]

# Only remove columns that actually exist
existing_columns_to_remove = [col for col in columns_to_remove if col in df.columns]

if existing_columns_to_remove:
    print(f"\nRemoving irrelevant columns: {existing_columns_to_remove}")
    df = df.drop(columns=existing_columns_to_remove)
else:
    print("\nNo irrelevant columns found to remove")

print(f"\nColumns after cleanup: {len(df.columns)}")
print("Remaining columns:")
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col}")

# Check for duplicate columns
duplicate_cols = df.columns[df.columns.duplicated()].tolist()
if duplicate_cols:
    print(f"\nWarning: Found duplicate columns: {duplicate_cols}")
else:
    print("\nNo duplicate columns found")

Current columns in dataset:
['ID', 'State', 'City', 'location', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Furnished_Status_encoded', 'Public_Transport_Accessibility_encoded', 'Parking_Space_encoded', 'Security_encoded', 'Availability_Status_encoded', 'Name', 'Property Title', 'Price', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony', 'balcony_encoded', 'Price_numeric', 'feature_importance', 'Date_x', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'Month_x', 'Year_x', 'AQI_Bucket_encoded', 'Year_y', 'Month_y', 'Day', 'Night', 'DayLimit', 'NightLimit', 'Date_y', 'StationEncoded', 'DayExcess', 'NightExcess', 'target_price']

Total c

In [16]:
# Enhanced Categorical Feature Encoding
from sklearn.preprocessing import LabelEncoder

# Check existing categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns found: {categorical_cols}")

# Remove target price from categorical if it exists
if 'target_price' in categorical_cols:
    categorical_cols.remove('target_price')

print(f"Columns to encode: {categorical_cols}")

# Initialize label encoders
label_encoders = {}

for col in categorical_cols:
    if col in df.columns:
        print(f"\nEncoding {col}:")
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Sample values: {df[col].dropna().head().tolist()}")
        
        # Handle missing values first
        df[col] = df[col].fillna('Unknown')
        
        # Apply label encoding
        le = LabelEncoder()
        df[f'{col}_encoded'] = le.fit_transform(df[col])
        label_encoders[col] = le
        
        print(f"  Encoded range: {df[f'{col}_encoded'].min()} to {df[f'{col}_encoded'].max()}")

# Check for furnishing-related columns and create binary features
furnishing_cols = [col for col in df.columns if 'furnish' in col.lower()]
if furnishing_cols:
    print(f"\nFurnishing columns found: {furnishing_cols}")
    
    # Create binary furnishing feature if not exists
    if 'furnishing_binary' not in df.columns:
        # Find the main furnishing column
        main_furnish_col = furnishing_cols[0]
        print(f"Using {main_furnish_col} for binary furnishing")
        
        # Create binary: 1 for furnished, 0 for unfurnished/semi-furnished
        furnished_values = ['furnished', 'fully furnished', 'full', 'yes']
        df['furnishing_binary'] = df[main_furnish_col].astype(str).str.lower().apply(
            lambda x: 1 if any(val in str(x) for val in furnished_values) else 0
        )
        print(f"Furnishing binary distribution: {df['furnishing_binary'].value_counts()}")

# Check for parking-related columns
parking_cols = [col for col in df.columns if 'park' in col.lower()]
if parking_cols:
    print(f"\nParking columns found: {parking_cols}")
    
    # Create binary parking feature if not exists
    if 'parking_binary' not in df.columns:
        main_parking_col = parking_cols[0]
        print(f"Using {main_parking_col} for binary parking")
        
        # Create binary: 1 for has parking, 0 for no parking
        df['parking_binary'] = df[main_parking_col].astype(str).str.lower().apply(
            lambda x: 1 if any(val in str(x) for val in ['yes', 'available', '1', 'true']) else 0
        )
        print(f"Parking binary distribution: {df['parking_binary'].value_counts()}")

print(f"\nDataset shape after encoding: {df.shape}")
print(f"New columns: {[col for col in df.columns if col.endswith('_encoded') or col.endswith('_binary')]}")

Categorical columns found: ['State', 'City', 'location', 'Property_Type', 'Furnished_Status', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Name', 'Property Title', 'Price', 'Balcony', 'Date_x', 'AQI_Bucket', 'Date_y']
Columns to encode: ['State', 'City', 'location', 'Property_Type', 'Furnished_Status', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Name', 'Property Title', 'Price', 'Balcony', 'Date_x', 'AQI_Bucket', 'Date_y']

Encoding State:
  Unique values: 1
  Sample values: ['Karnataka', 'Karnataka', 'Karnataka', 'Karnataka', 'Karnataka']
  Encoded range: 0 to 1

Encoding City:
  Unique values: 1
  Sample values: ['Bangalore', 'Bangalore', 'Bangalore', 'Bangalore', 'Bangalore']
  Encoded range: 0 to 1

Encoding location:
  Unique values: 7561
  Sample values: ['    Madhurangan Apartment ,Ambegaon, Pune', '   Manganahalli    Srir

In [17]:
# Environmental and Location-based Features

# Check for location-related columns
location_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['location', 'area', 'locality', 'city', 'address'])]
print(f"Location-related columns: {location_cols}")

# Check for environmental data columns
env_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['aqi', 'air', 'quality', 'pollution', 'noise', 'water'])]
print(f"Environmental columns: {env_cols}")

# Process location data
if location_cols:
    main_location_col = location_cols[0]
    print(f"\nProcessing location data from: {main_location_col}")
    
    # Get location frequency for popularity scoring
    location_counts = df[main_location_col].value_counts()
    print(f"Top 10 locations: {location_counts.head(10)}")
    
    # Create location popularity score
    df['location_popularity'] = df[main_location_col].map(location_counts)
    print(f"Location popularity range: {df['location_popularity'].min()} to {df['location_popularity'].max()}")

# Process environmental features
if env_cols:
    print(f"\nProcessing environmental features:")
    
    # Normalize environmental scores to 0-1 scale for consistency
    for col in env_cols:
        if df[col].dtype in ['int64', 'float64']:
            min_val = df[col].min()
            max_val = df[col].max()
            if max_val > min_val:
                df[f'{col}_normalized'] = (df[col] - min_val) / (max_val - min_val)
                print(f"  {col}: normalized from [{min_val}, {max_val}] to [0, 1]")
    
    # Create composite environmental score if multiple env features exist
    env_numeric_cols = [col for col in env_cols if df[col].dtype in ['int64', 'float64']]
    if len(env_numeric_cols) > 1:
        # Average of normalized environmental scores
        normalized_env_cols = [f'{col}_normalized' for col in env_numeric_cols if f'{col}_normalized' in df.columns]
        if normalized_env_cols:
            df['environmental_score'] = df[normalized_env_cols].mean(axis=1)
            print(f"Created composite environmental_score from {len(normalized_env_cols)} features")

# Create property size features if area columns exist
area_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['area', 'size', 'sqft', 'square'])]
if area_cols:
    main_area_col = area_cols[0]
    print(f"\nCreating size-based features from: {main_area_col}")
    
    if df[main_area_col].dtype in ['int64', 'float64']:
        # Property size categories
        area_quartiles = df[main_area_col].quantile([0.25, 0.5, 0.75])
        df['size_category'] = pd.cut(df[main_area_col], 
                                   bins=[-float('inf'), area_quartiles[0.25], area_quartiles[0.5], 
                                         area_quartiles[0.75], float('inf')],
                                   labels=['Small', 'Medium', 'Large', 'Extra_Large'])
        
        # Encode size category
        le_size = LabelEncoder()
        df['size_category_encoded'] = le_size.fit_transform(df['size_category'].astype(str))
        print(f"Size categories: {df['size_category'].value_counts()}")

print(f"\nNew environmental features created:")
new_env_features = [col for col in df.columns if any(suffix in col for suffix in ['_popularity', '_normalized', '_score', '_category'])]
for feat in new_env_features:
    print(f"  - {feat}")

print(f"\nDataset shape after environmental features: {df.shape}")

Location-related columns: ['City', 'location', 'Total_Area', 'City_encoded', 'location_encoded']
Environmental columns: ['AQI', 'AQI_Bucket', 'AQI_Bucket_encoded']

Processing location data from: City
Top 10 locations: City
Unknown      17242
Bangalore     4144
Name: count, dtype: int64
Location popularity range: 4144 to 17242

Processing environmental features:
  AQI: normalized from [20.0, 352.0] to [0, 1]
  AQI_Bucket_encoded: normalized from [0, 5] to [0, 1]
Created composite environmental_score from 2 features

Creating size-based features from: Size_in_SqFt
Size categories: size_category
Small          1038
Extra_Large    1036
Medium         1035
Large          1035
Name: count, dtype: int64

New environmental features created:
  - location_popularity
  - AQI_normalized
  - AQI_Bucket_encoded_normalized
  - environmental_score
  - size_category
  - size_category_encoded

Dataset shape after environmental features: (21386, 87)


In [18]:
# Advanced Numerical Feature Engineering

# Identify numerical columns (excluding target and encoded columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove target price and encoded columns from feature engineering
exclude_cols = ['target_price'] + [col for col in numeric_cols if col.endswith('_encoded') or col.endswith('_binary')]
numeric_features = [col for col in numeric_cols if col not in exclude_cols]

print(f"Numerical features for engineering: {numeric_features}")

# Create interaction features for key property characteristics
interaction_pairs = []

# Common real estate feature pairs
if any('bedroom' in col.lower() or 'bhk' in col.lower() for col in numeric_features):
    bedroom_cols = [col for col in numeric_features if 'bedroom' in col.lower() or 'bhk' in col.lower()]
    if bedroom_cols and any('area' in col.lower() or 'size' in col.lower() for col in numeric_features):
        area_cols = [col for col in numeric_features if 'area' in col.lower() or 'size' in col.lower()]
        if area_cols:
            interaction_pairs.append((bedroom_cols[0], area_cols[0]))

if any('bath' in col.lower() for col in numeric_features):
    bath_cols = [col for col in numeric_features if 'bath' in col.lower()]
    if bath_cols and any('bedroom' in col.lower() or 'bhk' in col.lower() for col in numeric_features):
        bedroom_cols = [col for col in numeric_features if 'bedroom' in col.lower() or 'bhk' in col.lower()]
        if bedroom_cols:
            interaction_pairs.append((bath_cols[0], bedroom_cols[0]))

print(f"Creating interaction features for pairs: {interaction_pairs}")

# Create interaction features
for col1, col2 in interaction_pairs:
    if col1 in df.columns and col2 in df.columns:
        # Multiplicative interaction
        df[f'{col1}_{col2}_interaction'] = df[col1] * df[col2]
        print(f"Created {col1}_{col2}_interaction")

# Create ratio features for meaningful property characteristics
ratio_features = []

# Area per bedroom ratio
bedroom_cols = [col for col in df.columns if 'bedroom' in col.lower() or 'bhk' in col.lower()]
area_cols = [col for col in df.columns if 'area' in col.lower() or 'size' in col.lower()]

if bedroom_cols and area_cols:
    bedroom_col = bedroom_cols[0]
    area_col = area_cols[0]
    
    # Avoid division by zero
    df[f'area_per_bedroom'] = df[area_col] / (df[bedroom_col] + 1)
    ratio_features.append('area_per_bedroom')
    print(f"Created area_per_bedroom ratio")

# Bathroom to bedroom ratio
bath_cols = [col for col in df.columns if 'bath' in col.lower()]
if bath_cols and bedroom_cols:
    bath_col = bath_cols[0]
    bedroom_col = bedroom_cols[0]
    
    df[f'bath_bedroom_ratio'] = df[bath_col] / (df[bedroom_col] + 1)
    ratio_features.append('bath_bedroom_ratio')
    print(f"Created bath_bedroom_ratio")

# Log transformation for skewed features (typically area and price-related)
log_candidates = []
for col in numeric_features:
    if col in df.columns and df[col].min() > 0:  # Only positive values for log
        skewness = df[col].skew()
        if abs(skewness) > 1:  # Highly skewed
            df[f'{col}_log'] = np.log1p(df[col])  # log1p handles zeros better
            log_candidates.append(f'{col}_log')
            print(f"Created log transform for {col} (skewness: {skewness:.2f})")

# Square root transformation for moderately skewed features
sqrt_candidates = []
for col in numeric_features:
    if col in df.columns and df[col].min() >= 0:  # Non-negative values for sqrt
        skewness = df[col].skew()
        if 0.5 < abs(skewness) <= 1:  # Moderately skewed
            df[f'{col}_sqrt'] = np.sqrt(df[col])
            sqrt_candidates.append(f'{col}_sqrt')
            print(f"Created sqrt transform for {col} (skewness: {skewness:.2f})")

print(f"\nNew numerical features created:")
new_numeric_features = (
    [f'{col1}_{col2}_interaction' for col1, col2 in interaction_pairs] +
    ratio_features + log_candidates + sqrt_candidates
)

for feat in new_numeric_features:
    if feat in df.columns:
        print(f"  - {feat}")

print(f"\nDataset shape after numerical feature engineering: {df.shape}")

Numerical features for engineering: ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Total_Area', 'Price_per_SQFT', 'Baths', 'Price_numeric', 'feature_importance', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'Month_x', 'Year_x', 'Year_y', 'Month_y', 'Day', 'Night', 'DayLimit', 'NightLimit', 'StationEncoded', 'DayExcess', 'NightExcess', 'location_popularity', 'AQI_normalized', 'AQI_Bucket_encoded_normalized', 'environmental_score']
Creating interaction features for pairs: [('BHK', 'Size_in_SqFt'), ('Baths', 'BHK')]
Created BHK_Size_in_SqFt_interaction
Created Baths_BHK_interaction
Created area_per_bedroom ratio
Created bath_bedroom_ratio
Created log transform for Price_numeric (skewness: 14.69)
Created log transform for PM2.5 (skewness: 3.47)
Created log transform for PM10 (skewness: 1.25)
Created log transform for NO (s

In [19]:
# Final Dataset Preparation and Export

# Remove original categorical columns that have been encoded
categorical_cols_to_remove = []
for col in df.columns:
    if f'{col}_encoded' in df.columns and col not in ['target_price']:
        categorical_cols_to_remove.append(col)

if categorical_cols_to_remove:
    print(f"Removing original categorical columns: {categorical_cols_to_remove}")
    df = df.drop(columns=categorical_cols_to_remove)

# Ensure target_price is present
if 'target_price' not in df.columns:
    print("Warning: target_price column not found!")
else:
    print(f"Target price statistics:")
    print(df['target_price'].describe())

# Remove any remaining object columns that couldn't be processed
object_cols = df.select_dtypes(include=['object']).columns.tolist()
if object_cols and 'target_price' not in object_cols:
    print(f"Removing remaining object columns: {object_cols}")
    df = df.drop(columns=object_cols)

# Handle missing values in the final dataset
print(f"\nMissing values before final cleanup:")
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
if len(missing_cols) > 0:
    print(missing_cols)
    
    # Fill missing values appropriately
    for col in missing_cols.index:
        if df[col].dtype in ['int64', 'float64']:
            # Fill numerical columns with median
            df[col] = df[col].fillna(df[col].median())
            print(f"Filled {col} with median: {df[col].median()}")
        else:
            # Fill categorical columns with mode
            mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
            df[col] = df[col].fillna(mode_val)
            print(f"Filled {col} with mode: {mode_val}")
else:
    print("No missing values found")

# Final dataset summary
print(f"\n=== FINAL DATASET SUMMARY ===")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Show column types
print(f"\nColumn types:")
for dtype in df.dtypes.value_counts().index:
    cols = df.select_dtypes(include=[dtype]).columns.tolist()
    print(f"  {dtype}: {len(cols)} columns")

# Save the final engineered dataset
output_path = r'c:\Users\drket\OneDrive\Desktop\Codes\RealEsatePredictor\feature_engineering\feature_engineered_expert.csv'
df.to_csv(output_path, index=False)
print(f"\nDataset saved to: {output_path}")

# Show final column list
print(f"\nFinal columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")
    
# Quick validation
print(f"\nQuick validation:")
print(f"Target price non-null: {df['target_price'].notna().sum()} / {len(df)}")
print(f"All columns have data: {(df.notna().sum() > 0).all()}")
print(f"No infinite values: {not np.isinf(df.select_dtypes(include=[np.number])).any().any()}")

print(f"\n✅ Feature engineering completed successfully!")

Removing original categorical columns: ['State', 'City', 'location', 'Property_Type', 'Furnished_Status', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Name', 'Property Title', 'Price', 'Balcony', 'Date_x', 'AQI_Bucket', 'Date_y', 'size_category']
Target price statistics:
count    1.451800e+04
mean     1.067542e+07
std      1.867913e+07
min      1.000000e+00
25%      3.700000e+06
50%      6.500000e+06
75%      1.140000e+07
max      8.400000e+08
Name: target_price, dtype: float64

Missing values before final cleanup:
BHK                             17242
Size_in_SqFt                    17242
Price_in_Lakhs                  17242
Price_per_SqFt                  17242
Year_Built                      17242
Floor_No                        17242
Total_Floors                    17242
Age_of_Property                 17242
Nearby_Schools                  17242
Nearby_Hospitals                17242
Total_Area          

C:\Users\drket\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\drket\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\drket\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\drket\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\drket\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\drket\AppData\Roaming\Pyth


Dataset saved to: c:\Users\drket\OneDrive\Desktop\Codes\RealEsatePredictor\feature_engineering\feature_engineered_expert.csv

Final columns (81):
 1. BHK
 2. Size_in_SqFt
 3. Price_in_Lakhs
 4. Price_per_SqFt
 5. Year_Built
 6. Floor_No
 7. Total_Floors
 8. Age_of_Property
 9. Nearby_Schools
10. Nearby_Hospitals
11. Furnished_Status_encoded
12. Public_Transport_Accessibility_encoded
13. Parking_Space_encoded
14. Security_encoded
15. Availability_Status_encoded
16. Total_Area
17. Price_per_SQFT
18. Baths
19. balcony_encoded
20. Price_numeric
21. feature_importance
22. PM2.5
23. PM10
24. NO
25. NO2
26. NOx
27. NH3
28. CO
29. SO2
30. O3
31. Benzene
32. Toluene
33. Xylene
34. AQI
35. Month_x
36. Year_x
37. AQI_Bucket_encoded
38. Year_y
39. Month_y
40. Day
41. Night
42. DayLimit
43. NightLimit
44. StationEncoded
45. DayExcess
46. NightExcess
47. target_price
48. State_encoded
49. City_encoded
50. location_encoded
51. Property_Type_encoded
52. Amenities_encoded
53. Facing_encoded
54. Owner_